# 02 — RDD e Spark SQL
Per entrambi i metodi leggiamo esclusivamente i tre Parquet puliti su HDFS indicati in `results/quality.json`. Il ricavo è la tariffa base e la velocità media è la media delle velocità individuali.

In [ ]:
import json
from collections import Counter

from src.aggregations import (
    INPUT_COLUMNS, TOLERANCES, aggregate_rdd, aggregate_sql, compare,
)
from src.config import INPUTS, RESULTS, save_json, spark_session

quality = json.loads((RESULTS / "quality.json").read_text(encoding="utf-8"))
aggregate_path = RESULTS / "aggregates.json"
aggregate_path.unlink(missing_ok=True)
spark = spark_session("TLC 02 - RDD e SQL")
frames = {
    category: spark.read.parquet(quality["categories"][category]["cleaned"]).select(*INPUT_COLUMNS)
    for category in INPUTS
}
common = frames["yellow"].unionByName(frames["green"]).unionByName(frames["hvfhv"])
rdd_rows = aggregate_rdd(common)
sql_rows = aggregate_sql(common)

## Equivalenza
Il confronto controlla l'insieme delle chiavi e i conteggi esatti. Per ricavo e medie si applica una tolleranza relativa di 10⁻¹⁰. Le tolleranze assolute sono 10⁻⁶ USD per il ricavo e 10⁻⁹ per distanza e velocità. Riportiamo la differenza assoluta massima osservata per metrica.

In [ ]:
comparison = compare(rdd_rows, sql_rows)
actual = Counter()
for row in sql_rows:
    actual[row.vehicle_type] += row.trip_count
expected = {category: quality["categories"][category]["counts"]["valid_count"] for category in INPUTS}
if any(actual[category] != count for category, count in expected.items()) or actual.keys() - expected.keys():
    comparison["problems"].append(f"Conteggi HDFS {dict(actual)} diversi dal report {expected}")
    comparison["verdict"] = "FAIL"

print(comparison)
if comparison["verdict"] != "PASS":
    raise RuntimeError("Verifica RDD/Spark SQL non superata")

aggregate_report = {
    "run_id": quality["run_id"],
    "comparison": comparison,
    "tolerances": TOLERANCES,
    "rdd": [row.asDict() for row in rdd_rows],
    "sql": [row.asDict() for row in sql_rows],
}
save_json(aggregate_path, aggregate_report)

## Benchmark
Per Green, Yellow e HVFHV materializziamo su disco locale lo stesso input per entrambi i metodi, una categoria alla volta. Dopo un warm-up per metodo misuriamo cinque coppie con ordine alternato. Il tempo comprende scansione della cache, trasformazioni, aggregazione, ordinamento e raccolta delle sole righe aggregate. La preparazione della cache, il preprocessing, le scritture su HDFS e PostgreSQL e il rendering rimangono fuori dal cronometro. Le misure descrivono questa macchina singola, con i suoi limiti di disco, CPU e RAM.

In [ ]:
from IPython.display import Image, Markdown, display
from src.benchmark import benchmark, save_benchmark

for name in ("benchmark.csv", "benchmark_summary.json", "benchmark.png"):
    (RESULTS / name).unlink(missing_ok=True)

timings, summary = benchmark(frames, quality["run_id"])
save_benchmark(timings, summary)

lines = [
    "| Categoria | Metodo | Media (s) | Mediana (s) | Dev. std. campionaria (s) |",
    "|---|---|---:|---:|---:|",
]
for category, stats in summary.items():
    for algorithm in ("rdd", "sql"):
        values = stats[algorithm]
        lines.append(
            f"| {category} | {algorithm} | {values['mean']:.3f} | "
            f"{values['median']:.3f} | {values['stdev']:.3f} |"
        )
display(Markdown("\n".join(lines)))
for category, stats in summary.items():
    ratio = stats["speedup_rdd_over_sql"]
    faster = "Spark SQL" if ratio > 1 else "RDD" if ratio < 1 else "pari"
    print(f"{category}: rapporto mediane RDD/SQL {ratio:.2f}; mediana minore: {faster}")
display(Image(filename=str(RESULTS / "benchmark.png")))

In [ ]:
spark.stop()